### Run mode
This notebook now defaults to a short debug run so logs appear quickly in the cell output.

1. Keep `DEBUG_MODE = True` for the first run.
2. After the smoke/debug run succeeds, change it to `False` for the 10-epoch short run.
3. The same output is printed in the notebook cell, Colab runtime log, and the Drive log file under `stage2/logs/`.


# Stage 2 joint experiment notebook

Run order after Exp2 refactor: 00 prepare dataset → 01 Exp2A baseline → 02 Exp2B GCA baseline → 03 Exp2D lane detail neck → 04 Exp2E matched lane loss → 05 Exp2F detail + matching. Run 06 KD only if the teacher checkpoint exists. Run Exp3 notebooks only after Exp2F improves lane geometry. Every notebook re-extracts the Drive tar into `/content`; never assume files from a previous Colab runtime still exist. Training logs are printed in this notebook cell and mirrored to `/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/`.


# Stage 2 Notebook 05 - Exp2F lane detail neck + matched lane loss

This combines the two likely fixes: high-resolution lane detail features and Hungarian-style lane matching. This is the main candidate for the next serious Exp2 run.

This notebook prints training output directly in the cell and mirrors the same text to Drive logs under `/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/`.


In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)


Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys
CONFIG = 'stage2/configs/exp05_rmt_gca_lane_detail_matching_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)


[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp05_rmt_gca_lane_detail_matching_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp05_rmt_gca_lane_detail_matching_joint_smoke.log
OK exp05_rmt_gca_lane_detail_matching_joint.yaml
  lane_shape=(1, 2, 72, 2) det_shape=(1, 4, 4)
  lane_loss=3.1884 det_loss=3.4502 grad_cos=0.1503 lambda_lane=0.1109
  gate_stats={'gate/det_mean': 0.5016847252845764, 'gate/lane_mean': 0.5021314024925232, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0, 'gate/p3_det_mean': 0.5017589926719666, 'gate/p3_lane_mean': 0.49687960743904114, 'gate/p3_det_sat_low': 0.0, 'gate/p3_det_sat_high': 0.0, 'gate/p3_lane_sat_low': 0.0, 'gate/p3_lane_sat_high': 0.0, 'gate/p4_det_mean': 0.5001861453056335, 'gate/p4_lane_mean': 0.5022441744804382, 'gate/p4_det_sat_low': 0.0, 'gate/p4_det_sat_high': 0.0, 'gate/p4_lane_sat_low'

0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp05_rmt_gca_lane_detail_matching_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

# First run should be a small debug run. Change to False only after logs,
# smoke test, and metrics look normal.
DEBUG_MODE = True

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)


DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp05_rmt_gca_lane_detail_matching_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp05_rmt_gca_lane_detail_matching_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp05_rmt_gca_lane_detail_matching_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp05_rmt_gca_lane_detail_matching_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp05_rmt_gca_lane_detail_matching_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp05_rmt_gca_lane_detail_matching_joint.yaml --curve-tar /content/drive/MyDrive

0

## What to watch during training

- `val/lane_point_mae`: lower is better. This is the main geometry signal for the current Exp2 debugging.
- `val/lane_exist_acc`: checks whether lane existence is learned. High existence with flat MAE means geometry is still weak.
- `val/det_loss`: lower is better for vehicle detection.
- `train/mtl/lambda_lane_runtime` and `train/mtl/lambda_lane_epoch`: show the lane weight used in the joint loss.
- `gate/p3_lane_mean`, `gate/p4_lane_mean`, `gate/p5_lane_mean`: only in detail/GCA runs; these show how strongly the lane branch uses task-specific features per scale.
